# Python in a hurry {#sec-python}

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/waelrash1/loopbricks/blob/book-ai/notebooks/a01-python.ipynb)

Every chapter of this book is a notebook of Python code. This appendix teaches the Python the chapters use, and no more. You get the core language, then the three libraries that do the heavy lifting: **NumPy** for arrays of numbers, **pandas** for tables and time series, and **matplotlib** for charts. Every example uses the waiting list from the running case, so nothing here is wasted.

**Who it is for.** Readers who have never written Python, or who last wrote it years ago. If you know another language (R, MATLAB, Excel formulas, Vensim equations), the ideas will be familiar and only the spelling is new.

**Who can skip what.** If you write Python every week, skim the quick reference in @sec-py-reference and go back to the chapters. If you know Python but not pandas, start at @sec-py-pandas.

**How long.** About three hours if you run every cell and try every exercise. One hour to read it through.

**How to use it.** Open it in Colab with the badge above, or in Jupyter on your own machine. Run each cell with **Shift+Enter** and read the output before moving on. Change the numbers and run the cell again: you cannot break anything, and if a cell gets into a mess, *Runtime → Restart* (Colab) or *Kernel → Restart* (Jupyter) and run the cells from the top.

## Learning outcomes

By the end of this appendix you will be able to:

1. **Write** Python with variables, lists, dictionaries, loops and functions, and **read** the error message when something goes wrong (Try it 1 to 4).
2. **Compute** with NumPy arrays without writing loops (Try it 5).
3. **Load**, select, summarise and reshape a time series with pandas (Try it 6).
4. **Draw** a labelled chart with matplotlib (Try it 7).
5. **Build** a stock-and-flow model three ways, in plain Python, with NumPy and with the `loopbricks` package, and **check** that they agree (Try it 8).

In [ ]:
# Setup: run this cell first. In Colab it installs loopbricks (ten seconds).
try:
    import loopbricks as lb
except ImportError:
    import subprocess, sys
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q",
                           "loopbricks @ git+https://github.com/waelrash1/"
                           "loopbricks.git@book-ai#subdirectory=python"])
    import loopbricks as lb
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

DATA = "data/waiting_list.csv" if os.path.exists("data/waiting_list.csv") else \
    "https://raw.githubusercontent.com/waelrash1/loopbricks/book-ai/notebooks/data/waiting_list.csv"
print("loopbricks", lb.__version__, "| numpy", np.__version__,
      "| pandas", pd.__version__)

The setup cell does three things you will see at the top of every chapter. It **imports** the libraries, giving each a short nickname (`np`, `pd`, `plt`, `lb`) that everyone uses. If `loopbricks` is missing, it installs it first: `try` attempts something and `except` says what to do if it fails (@sec-py-errors). And it works out where the data file is, on your disk or on GitHub.

## 1. Values, names and arithmetic

Python is a calculator first. Type an expression and the notebook shows its value.

In [ ]:
4000 - 3950

Give a value a **name** with `=` and use the name from then on. Names are case sensitive, cannot contain spaces and are written in `lower_case_with_underscores` by convention. A `#` starts a **comment**: Python ignores the rest of the line, so comments are for the reader.

In [ ]:
referral_rate = 4000      # patients a month
capacity = 3950           # patients a month
net_flow = referral_rate - capacity
waiting_list = 32000      # patients
months = 12
waiting_list + net_flow * months

`=` means *make this name point at this value*, not *these are equal*. So `waiting_list = waiting_list + net_flow` is a perfectly good line: work out the right-hand side with the old value, then move the name to the result. That one line is the whole idea of simulating a **stock**, a quantity that accumulates, like the patients on the list: each step adds what came in and takes away what went out. Chapter 1 builds on it, and you will meet it again in @sec-py-model.

The arithmetic operators are the usual ones, with two to watch:

| Operator | Meaning | Example | Result |
|---|---|---|---|
| `+ - * /` | add, subtract, multiply, divide | `7 / 2` | `3.5` |
| `**` | power (not `^`) | `1.05 ** 10` | `1.628...` |
| `//` | divide and round down | `7 // 2` | `3` |
| `%` | remainder | `7 % 2` | `1` |

In [ ]:
print(7 / 2, 7 // 2, 7 % 2, 2 ** 10)
print(1.05 ** 10)               # ten years of 5% growth
print(0.1 + 0.2)                # not quite 0.3: see the box below
print(round(0.1 + 0.2, 10) == 0.3)

**Why 0.1 + 0.2 is not 0.3.** Computers store numbers in binary, where 0.1 cannot be written exactly, just as 1/3 cannot be written exactly in decimals. The error is tiny, in the 17th digit, but it means you should never test whether two computed numbers are exactly equal. Compare with a tolerance instead: `abs(a - b) < 1e-9`, or `np.isclose(a, b)` from @sec-py-numpy. `1e-9` is scientific notation for 0.000000001.

`print` shows several values on one line. A cell also shows the value of its last line on its own, which is why the first cells needed no `print`.

### Types

Every value has a **type**. The ones you need are whole numbers (`int`), decimal numbers (`float`), text (`str`), true or false (`bool`) and the empty value `None`.

In [ ]:
for value in [4000, 0.125, "Waiting list", True, None]:
    print(repr(value), "is a", type(value).__name__)

Text goes in quotes, single or double. Comparisons give `bool` values, and `and`, `or` and `not` combine them:

In [ ]:
print(net_flow > 0, net_flow == 50, net_flow != 50)
print(net_flow > 0 and waiting_list > 30000)
print("list" in "Waiting list")       # is one string inside another?

Note the double `==` for *is equal to*. A single `=` there is an error, because it means assignment.

### Formatted text

The chapters print results with **f-strings**: put `f` before the opening quote, and anything in `{curly brackets}` is worked out and put into the text. After a colon you can say how to format a number.

In [ ]:
growth = 0.0123456
total = 69927
print(f"The list ends at {total} patients.")
print(f"With a thousands separator: {total:,} patients.")
print(f"Two decimals: {growth:.2f}; as a percentage: {growth:.1%}; "
      f"with a sign: {-growth:+.1%}")
print(f"Right-aligned in 8 characters: [{total:>8,}]")

| Format | Meaning | `{1234.567:...}` gives |
|---|---|---|
| `.2f` | two decimals | `1234.57` |
| `,.0f` | thousands separator, no decimals | `1,235` |
| `.1%` | multiply by 100, one decimal, add `%` | `123456.7%` |
| `+.1f` | always show the sign | `+1234.6` |
| `>10` | right-align in 10 characters | `  1234.567` |
| `.3g` | three significant figures | `1.23e+03` |

**Try it 1.**

- *Check.* At 4,000 referrals and 3,950 treatments a month, how many patients does the list gain in a year? Print it with a thousands separator.
- *Apply.* If referrals grow by 3% a year, how many will there be a month in ten years? Use `**`.
- *Extend.* Print `0.1 + 0.2 == 0.3` and then a version that compares with a tolerance and gives `True`.

<details><summary>Worked answer</summary>

```python
print(f"{(4000 - 3950) * 12:,} patients a year")      # 600
print(f"{4000 * 1.03 ** 10:,.0f} referrals a month")  # 5,376
print(0.1 + 0.2 == 0.3, abs((0.1 + 0.2) - 0.3) < 1e-9)
```

</details>

## 2. Collections: lists, tuples and dictionaries

A **list** holds values in order, in square brackets. Positions count from **zero**, and negative positions count from the end.

In [ ]:
monthly = [32276, 32311, 32611, 32824, 33070, 33191]   # first six months of 2014
print(monthly[0], monthly[1], monthly[-1])   # first, second, last
print(len(monthly), "months, total", sum(monthly), "max", max(monthly))

A **slice** `start:stop` takes the items from `start` up to but **not including** `stop`. Leave out either end to go to the start or the end. This half-open rule takes getting used to, but it means `x[:3]` and `x[3:]` split a list with no overlap and `len(x[a:b])` is `b - a`.

In [ ]:
print(monthly[:3])     # first three
print(monthly[3:])     # the rest
print(monthly[1:4])    # positions 1, 2 and 3
print(monthly[-2:])    # last two
print(monthly[::2])    # every second one

Lists can change: add with `append`, replace by position. A **tuple** is a list in round brackets that cannot change; you meet tuples mostly when a function returns several values. `range(n)` gives the whole numbers `0` to `n - 1`.

In [ ]:
monthly.append(32950)
monthly[0] = 32000
print(monthly)
point = (2014, 32276)               # a tuple
year, count = point                 # unpack it into two names
print(year, count, list(range(5)))

A **dictionary** (`dict`) looks values up by a **key** instead of a position. The chapters use dictionaries for model settings and for collecting results.

In [ ]:
params = {"referral_rate": 4000, "capacity": 3950,
          "minimum_treatment_time": 0.5}
print(params["capacity"])
params["capacity"] = 4100           # change one value
params["dt"] = 0.125                # add a new key
print(params)
print(list(params.keys()))
print(params.get("missing", "no such key"))   # a default instead of an error

**Lists share, they do not copy.** `b = a` makes `b` a second name for the *same* list, so changing `b` changes `a`. Use `b = a.copy()` (or `copy.deepcopy(a)` for lists of lists) when you want an independent copy. This catches everyone once.

In [ ]:
a = [1, 2, 3]
b = a
c = a.copy()
b.append(4)
print(a, b, c)

**Try it 2.**

- *Check.* From `monthly`, take the last three values with a slice.
- *Apply.* Make a dictionary with the keys `"Inventory"` and `"Supply line"` and the values 60 and 60, then add 5 to the inventory.
- *Extend.* Predict what `monthly[2:2]` gives, then run it.

<details><summary>Worked answer</summary>

```python
print(monthly[-3:])
stocks = {"Inventory": 60, "Supply line": 60}
stocks["Inventory"] += 5           # same as stocks["Inventory"] = stocks["Inventory"] + 5
print(stocks)
print(monthly[2:2])                # [] : from 2 up to but not including 2 is nothing
```

</details>

## 3. Decisions and loops

Python groups lines by **indentation**, not by brackets or `end`. A line ending in a colon opens a block, and every line of the block is indented by the same four spaces. Getting the indentation wrong is the commonest beginner's error, and Python says so with `IndentationError`.

In [ ]:
waiting_list = 32000
if waiting_list > 50000:
    print("Critical")
elif waiting_list > 30000:
    print("High")
else:
    print("Normal")
print("This line is outside the if, so it always runs.")

A **for loop** runs its block once for each item of a list, a range, the keys of a dictionary or the rows of anything that can be walked through. `enumerate` gives you the position as well, and `zip` walks two lists side by side.

In [ ]:
for month in range(3):
    waiting_list = waiting_list + net_flow
    print("month", month + 1, "list", waiting_list)

for key, value in params.items():
    print(f"{key:>24} = {value}")

for i, (label, value) in enumerate(zip(["Jan", "Feb", "Mar"], monthly)):
    print(i, label, value)

A **while loop** runs as long as a condition is true; use it when you do not know in advance how many steps you need. `break` leaves a loop early.

In [ ]:
stock, month = 100.0, 0
while stock > 50:          # how long until half has drained at 10% a month?
    stock = stock - 0.1 * stock
    month += 1
print(f"Below 50 after {month} months ({stock:.1f} left)")

A **comprehension** builds a list in one line from a loop. Read `[expression for item in items if condition]` as *the list of expression, for each item, keeping those that pass the condition*. The chapters use them often for small tables.

In [ ]:
growth_factors = [1.03 ** year for year in range(6)]
print([round(g, 3) for g in growth_factors])
print([m for m in monthly if m > 32700])
squares = {n: n * n for n in range(5)}           # a dictionary comprehension
print(squares)

**Try it 3.**

- *Check.* Write a loop that prints the years 2014 to 2023.
- *Apply.* Starting from a list of 32,000 that gains 50 patients a month, use a `while` loop to find the month it passes 33,000.
- *Extend.* With a comprehension, make the list of the monthly changes of `monthly` (each value minus the one before).

<details><summary>Worked answer</summary>

```python
for year in range(2014, 2024):     # stop is not included, so 2024
    print(year)

wl, m = 32000, 0
while wl <= 33000:
    wl += 50
    m += 1
print(m)                           # 21: 32,000 + 21 × 50 = 33,050

changes = [monthly[i] - monthly[i - 1] for i in range(1, len(monthly))]
print(changes)
```

</details>

## 4. Functions

A **function** packages lines you want to run more than once. `def` names it and lists its **parameters**, the indented block does the work, and `return` hands back the result. The text in triple quotes under `def` is a **docstring**: it says what the function does, and `help(name)` shows it.

In [ ]:
def treatments(waiting_list, capacity, minimum_treatment_time=0.5):
    """Patients treated a month: capacity, unless the list is too short."""
    return min(capacity, waiting_list / minimum_treatment_time)

print(treatments(32000, 3950))          # plenty waiting: capacity binds
print(treatments(1000, 3950))           # short list: at most 1000 / 0.5
print(treatments(1000, 3950, minimum_treatment_time=0.25))

Parameters with an `=` in the `def` line have a **default**, so you can leave them out. When you call a function you can pass arguments by position or by name; by name is clearer when there are several numbers, and it is how the chapters call `run(dt=0.25)`.

Two pieces of syntax that look odd at first:

- `**params` in a call spreads a dictionary into named arguments, so `treatments(**{"waiting_list": 1000, "capacity": 3950})` is the same as `treatments(waiting_list=1000, capacity=3950)`. The chapters use it to pass a whole set of settings to a model.
- `lambda x: ...` makes a small function in one line, with no name. It is handy as an argument, for example to say how to sort.

In [ ]:
setting = {"waiting_list": 1000, "capacity": 3950}
print(treatments(**setting))

scenarios = {"base": 3950, "extra clinics": 4200, "cuts": 3700}
for name, cap in sorted(scenarios.items(), key=lambda item: item[1]):
    print(f"{name:>14}: capacity {cap}")

A function can return several values as a tuple, and the caller can unpack them. Names made inside a function stay inside it: they do not change names of the same spelling outside.

In [ ]:
def flows(waiting_list, referral_rate=4000, capacity=3950):
    """Return the inflow, the outflow and the net flow."""
    inflow = referral_rate
    outflow = treatments(waiting_list, capacity)
    return inflow, outflow, inflow - outflow

r, t, net = flows(32000)
print(r, t, net)

**Try it 4.**

- *Check.* Call `flows` for a list of 1,500 patients. Which flow is the bottleneck?
- *Apply.* Write a function `doubling_time(rate)` that returns `np.log(2) / rate`, and check that 3% a year doubles in about 23 years.
- *Extend.* Give `doubling_time` a docstring and read it back with `help(doubling_time)`.

<details><summary>Worked answer</summary>

```python
print(flows(1500))     # (4000, 3000.0, 1000.0): 1,500 / 0.5 = 3,000 < capacity, so the list binds

def doubling_time(rate):
    """Years to double at a constant fractional growth rate a year."""
    return np.log(2) / rate

print(doubling_time(0.03))   # 23.1
help(doubling_time)
```

</details>

## 5. When things go wrong {#sec-py-errors}

Errors are normal and the messages are helpful once you know how to read them. Read the **last line first**: it names the kind of error and what went wrong. The lines above it show where, with the line of your code marked by an arrow.

| Error | Usual cause |
|---|---|
| `NameError: name 'x' is not defined` | a typo, or a cell that defines `x` has not been run yet |
| `SyntaxError` | a missing bracket, colon or quote |
| `IndentationError` | lines of one block not lined up |
| `TypeError` | the wrong kind of value, such as adding text to a number |
| `KeyError` | a dictionary key, or a pandas column, that does not exist |
| `IndexError` | a list position past the end |
| `ZeroDivisionError` | dividing by zero |
| `ModuleNotFoundError` | a library that is not installed; run the setup cell |

`try` and `except` let you catch an error you expect and carry on. Here we catch the error and print its message, which is the same message Python would have shown.

In [ ]:
try:
    params["capcity"]                 # a typo
except KeyError as error:
    print("KeyError:", error)

try:
    "Waiting list: " + 32000
except TypeError as error:
    print("TypeError:", error)
    print("Fix: " + "Waiting list: " + str(32000))

`assert` is the opposite: it stops with an error if something you believe is **not** true. The chapters use it to build checks into a model, such as *the stock never goes negative* or *two methods agree*. An assert that passes prints nothing.

In [ ]:
assert treatments(0, 3950) == 0, "an empty list cannot be treated"
assert abs(0.1 + 0.2 - 0.3) < 1e-9
print("both checks passed")

**Notebook trap: running cells out of order.** A notebook remembers every name from every cell you have run, in the order you ran them, not the order on the page. If results look impossible, restart and run all cells from the top. The chapters are written to run top to bottom.

## 6. NumPy: arrays of numbers {#sec-py-numpy}

A model run produces hundreds of numbers, one per time step. Looping over them one by one works but is slow and wordy. A NumPy **array** holds many numbers of one type, and arithmetic on an array applies to every element at once. This is called **vectorisation**, and it is why the chapters rarely need loops outside the simulation itself.

In [ ]:
counts = np.array([32276, 32311, 32611, 32824, 33070, 33191])
print(counts / 1000)                  # every element divided
print(counts - counts[0])             # change since the first month
print(counts.mean(), counts.min(), counts.max(), counts.sum())
print(counts.shape, counts.dtype)

Compare a list: `[1, 2] * 2` repeats the list to `[1, 2, 1, 2]`, while `np.array([1, 2]) * 2` doubles each number to `[2, 4]`. If arithmetic on a list does something strange, you probably wanted an array.

### Making arrays

`np.arange(start, stop, step)` is `range` for decimals (stop not included); `np.linspace(start, stop, n)` gives exactly `n` evenly spaced points including both ends. For model time, `linspace` is safer, because adding up small decimal steps can land a hair short of or past the end.

In [ ]:
t = np.arange(0, 2, 0.25)
print(t)
t = np.linspace(0, 2, 9)
print(t)
print(np.zeros(3), np.ones(3), np.full(3, 7.0))

### Functions on whole arrays

NumPy has a vectorised version of every mathematical function: `np.exp`, `np.log` (natural log), `np.sqrt`, `np.abs`, `np.sin`, plus `np.minimum` and `np.maximum` for element-by-element min and max. These let you write a model's analytic solution in one line and compare it with a simulation.

In [ ]:
years = np.arange(0, 61, 10)
population = 100 * np.exp(0.03 * years)        # exponential growth at 3%
print(np.round(population, 1))
print(np.minimum(population, 300))             # cap each value at 300
print(np.log(2) / 0.03)                        # doubling time

### Selecting with conditions

A comparison on an array gives an array of `True` and `False`, a **mask**. Use the mask in square brackets to keep the elements where it is true. `np.where(condition, a, b)` picks `a` where the condition holds and `b` elsewhere. `.sum()` on a mask counts the trues. `np.argmax` gives the position of the largest value, and on a mask that is the position of the first `True`: the first time something happens. (`np.argmin` is the position of the smallest.)

In [ ]:
mask = population > 200
print(mask)
print(population[mask])                         # only the values above 200
print("years above 200:", years[mask])
print("how many:", mask.sum())
print(np.where(population > 200, "high", "ok"))
print("first year above 200:", years[np.argmax(mask)])

### Changes and running totals

`np.diff` gives the change from each element to the next, which is the **net flow** of a stock series: inflow minus outflow. `np.cumsum` gives the running total, which turns a flow back into a stock. They undo each other, which is the bathtub idea of chapter 1 in two functions. Below, `np.concatenate` joins arrays end to end (here a 0 in front, for the first month), and `np.array_equal` checks that two arrays match element by element.

In [ ]:
net = np.diff(counts)
print("net flows:", net)
rebuilt = counts[0] + np.concatenate([[0], np.cumsum(net)])
print("rebuilt:  ", rebuilt)
print("same as the data:", np.array_equal(rebuilt, counts))

### Tables of numbers and random numbers

Arrays can have two dimensions, rows and columns. `a[i, j]` is one element, `a[:, j]` a whole column, and `axis=0` makes a summary run down the columns. For randomness, make one **generator** with a fixed seed, so that every run of the notebook gives the same numbers. Each chapter does this once at the top with `rng = np.random.default_rng(SEED)`.

In [ ]:
rng = np.random.default_rng(1)
runs = rng.normal(loc=4000, scale=200, size=(1000, 12))   # 1,000 years of monthly referrals
print(runs.shape)
yearly = runs.sum(axis=1)                                   # total per year
print(f"mean {yearly.mean():,.0f}, 5th to 95th percentile "
      f"{np.percentile(yearly, 5):,.0f} to {np.percentile(yearly, 95):,.0f}")

**Try it 5.**

- *Check.* Make an array of the decay of a stock of 100 with a decay time of 5 (the average time a unit stays in the stock), `100 * np.exp(-t / 5)`, for `t` from 0 to 20 in steps of 1, and find how much is left at `t = 5`.
- *Apply.* Use a mask to find the first time the stock is below 10.
- *Extend.* Draw 10,000 values from `rng.uniform(3, 8, size=10000)` as uncertain decay times and find the share of them for which less than 10 is left at `t = 15`.

<details><summary>Worked answer</summary>

```python
t = np.arange(0, 21)
s = 100 * np.exp(-t / 5)
print(s[5])                          # 36.8: about 37% is left after one decay time
print(t[np.argmax(s < 10)])          # 12
taus = rng.uniform(3, 8, size=10000)
print((100 * np.exp(-15 / taus) < 10).mean())   # the mean of a mask is a share
```

</details>

## 7. pandas: tables and time series {#sec-py-pandas}

Real data comes as tables with named columns and dates. **pandas** gives you two objects: a **Series** is one column of values with labels (the **index**), and a **DataFrame** is a table of Series that share an index. Model runs from `loopbricks` come back as DataFrames indexed by time, so everything in this section works on them too.

### Loading and looking

In [ ]:
wl = pd.read_csv(DATA, parse_dates=["month"])
print(wl.shape)            # rows, columns
wl.head()

`read_csv` reads a comma-separated file; `parse_dates` turns the `month` column from text into real dates. `head()` shows the first five rows and `tail()` the last. `info()` lists the columns and their types, and `describe()` summarises each numeric column.

In [ ]:
wl.drop(columns="month").describe().round(0)

The columns are the total waiting and four bands of how long patients have waited, in months: under 3, 3 to 6, 6 to 12 and over 12.

### Selecting columns and rows

One column is `wl["total"]`, a Series; several are `wl[["total", "w12_plus"]]`, a DataFrame. To pick rows, use `.loc` with **labels** and `.iloc` with **positions**. The rule is easy to forget, so say it as: `iloc` is for *integer location*. `.iloc[-1]` is the last row, whatever its label.

In [ ]:
wl = wl.set_index("month")          # make the dates the index
total = wl["total"]
print(total.iloc[0], total.iloc[-1])          # first and last by position
print(total.loc["2020-01-01"])                # one month by label
print(total.loc["2020"].tail(3))              # every month of 2020, last three shown

With a date index, a string like `"2020"` or `"2022":"2023"` selects a year or a range of years. Unlike positions, **label slices include both ends**.

Conditions work as in NumPy, and you can combine them with `&` (and) and `|` (or), with brackets round each part.

In [ ]:
long_waits = wl["w12_plus"] / wl["total"]           # share waiting over a year
print(long_waits.round(2).iloc[[0, -1]])
print("months over 45%:", (long_waits > 0.45).sum())
wl[(wl["total"] > 60000) & (long_waits < 0.45)].head(3)

### New columns and changes over time

Add a column by assigning to a new name. `.diff()` is the change since the previous row, `.shift(1)` moves values down a row, `.pct_change()` is the change as a share and `.rolling(12).mean()` a twelve-month moving average. All of them keep the dates lined up.

In [ ]:
wl["net_flow"] = wl["total"].diff()
wl["share_12_plus"] = wl["w12_plus"] / wl["total"]
wl["trend"] = wl["total"].rolling(12).mean()
wl[["total", "net_flow", "share_12_plus", "trend"]].iloc[10:14].round(2)

The first row of `net_flow` is `NaN`, *not a number*: there is no previous month to subtract. pandas uses `NaN` for missing values and skips it in sums and means. `dropna()` removes the rows that have it, and `fillna(0)` replaces it.

### Summaries by group

`groupby` splits the rows into groups, applies a summary to each and puts the results together. Here we take the average net flow for each year, the table chapter 1 reads to show that the list grew slowly, then jumped in 2020.

In [ ]:
by_year = wl["net_flow"].groupby(wl.index.year).mean()
print(by_year.round(0).to_string())

`to_string()` prints the whole Series without the summary pandas adds at the bottom. `resample("YE")` does the same job by calendar period for a date index: `wl["total"].resample("YE").last()` is the list at each year end.

### Building tables

You can make a DataFrame from a dictionary of columns. This is how the chapters collect the results of several runs into one table to print or plot.

In [ ]:
rows = []
for cap in [3900, 3950, 4000, 4050]:
    rows.append({"capacity": cap, "net flow": 4000 - cap,
                 "list after 10 years": 32000 + (4000 - cap) * 120})
table = pd.DataFrame(rows).set_index("capacity")
print(table.to_string())

**Try it 6.**

- *Check.* What was the total in December 2023, and what share of it had waited over a year?
- *Apply.* In how many months of 2022 and 2023 did the list fall?
- *Extend.* Make a table with one row a year and two columns, the year-end total and the year's largest monthly rise.

<details><summary>Worked answer</summary>

```python
print(wl.loc["2023-12-01", "total"], round(wl.loc["2023-12-01", "share_12_plus"], 2))
print((wl.loc["2022":"2023", "net_flow"] < 0).sum())
yearly = pd.DataFrame({
    "year-end total": wl["total"].groupby(wl.index.year).last(),
    "largest rise": wl["net_flow"].groupby(wl.index.year).max(),
})
print(yearly.to_string())
```

</details>

## 8. matplotlib: charts

Every chart in the book starts the same way: `fig, ax = plt.subplots()` makes a figure (`fig`) with one set of axes (`ax`) to draw on. Then you draw with methods of `ax`, label it, and show it.

In [ ]:
fig, ax = plt.subplots()
ax.plot(wl.index, wl["total"] / 1000, label="Patients waiting")
ax.plot(wl.index, wl["trend"] / 1000, ls="--", label="12-month average")
ax.axvline(pd.Timestamp("2020-03-01"), color="grey", ls=":", lw=1)   # one date
ax.set_title("Outpatient waiting list")
ax.set_xlabel("Month")
ax.set_ylabel("Patients waiting (thousands)")
ax.legend()
plt.show()

The pieces you will see most:

| Call | What it does |
|---|---|
| `ax.plot(x, y, label=..., ls="--", lw=2, color=...)` | a line |
| `ax.bar(x, heights)`, `ax.scatter(x, y)`, `ax.hist(values, bins=30)` | bars, points, a histogram |
| `ax.fill_between(x, low, high, alpha=0.3)` | a shaded band, used for uncertainty |
| `ax.axhline(y)`, `ax.axvline(x)` | a reference line across the chart |
| `ax.set(title=..., xlabel=..., ylabel=..., xlim=..., ylim=...)` | all the labels in one call |
| `ax.legend()`, `ax.text(x, y, "note")` | the key, a note on the chart |

A pandas Series can plot itself onto an axis, using its index for x: `series.plot(ax=ax, label=...)`. For several panels, ask for a grid: `fig, axes = plt.subplots(1, 2, figsize=(10, 4))` gives two axes side by side, `axes[0]` and `axes[1]`.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(10, 4))
wl["net_flow"].plot(ax=axes[0])
axes[0].axhline(0, color="black", lw=0.8)
axes[0].set(title="Net flow", ylabel="patients a month")
wl["share_12_plus"].plot(ax=axes[1])
axes[1].set(title="Share waiting over a year", ylim=(0, 0.6))
plt.tight_layout()       # stop the panels overlapping
plt.show()

**Try it 7.**

- *Check.* Plot the four waiting-time bands (`w0_3` to `w12_plus`) on one chart with a legend.
- *Apply.* Draw `by_year` as a bar chart with a line at zero.
- *Extend.* Use `ax.fill_between` to shade between the 5th and 95th percentiles of the `runs` array from section 6, month by month.

<details><summary>Worked answer</summary>

```python
fig, ax = plt.subplots()
for col in ["w0_3", "w3_6", "w6_12", "w12_plus"]:
    ax.plot(wl.index, wl[col], label=col)
ax.legend(); plt.show()

fig, ax = plt.subplots()
ax.bar(by_year.index, by_year); ax.axhline(0, color="black"); plt.show()

fig, ax = plt.subplots()
m = np.arange(1, 13)
ax.fill_between(m, np.percentile(runs, 5, axis=0), np.percentile(runs, 95, axis=0), alpha=0.3)
ax.plot(m, runs.mean(axis=0)); plt.show()
```

</details>

## 9. One model, three ways {#sec-py-model}

Now put it together. The waiting-list model of chapter 1 has one stock and two flows: referrals arrive at a constant rate, and treatments run at capacity unless the list is too short. **Euler's method** simulates it by repeating three steps: work out the flows from the stock as it is now, move the stock by the net flow times the time step `dt`, and move time on by `dt`.

**First, plain Python**, with a loop and lists. This is the clearest version and the one to write when you are unsure.

In [ ]:
def simulate(T=24, dt=0.125, waiting_list=32000.0, referral_rate=4000.0,
             capacity=3950.0, minimum_treatment_time=0.5):
    """Euler's method for the one-stock waiting list. Time in months."""
    steps = round(T / dt)
    times, stock = [], []
    for i in range(steps + 1):
        times.append(i * dt)
        stock.append(waiting_list)
        referrals = referral_rate
        treatments = min(capacity, waiting_list / minimum_treatment_time)
        waiting_list = waiting_list + dt * (referrals - treatments)
    return pd.Series(stock, index=times, name="Waiting list")

plain = simulate()
print(plain.iloc[[0, 8, -1]])

**Second, NumPy**, with the time steps made in one go and the results kept in an array. `np.asarray` turns a number or a list into an array (and leaves an array alone), and `np.empty` makes an array of a given shape to fill in. The loop is still there, because each step needs the one before, but storage and analysis are vectorised. Running several capacities at once, as one array, is where this version pays off.

In [ ]:
def simulate_np(T=24, dt=0.125, waiting_list=32000.0, referral_rate=4000.0,
                capacity=3950.0, minimum_treatment_time=0.5):
    """The same model; capacity may be an array to run many settings at once."""
    times = np.linspace(0, T, round(T / dt) + 1)
    capacity = np.asarray(capacity, dtype=float)
    stock = np.empty((len(times),) + capacity.shape)
    stock[0] = waiting_list
    for i in range(len(times) - 1):
        treatments = np.minimum(capacity, stock[i] / minimum_treatment_time)
        stock[i + 1] = stock[i] + dt * (referral_rate - treatments)
    return times, stock

times, stock = simulate_np()
print("largest difference from plain Python:",
      np.abs(stock - plain.to_numpy()).max())

caps = np.array([3900, 3950, 4000, 4050])
times, many = simulate_np(capacity=caps)
print(pd.DataFrame(many[[0, -1]], index=["start", "month 24"],
                   columns=caps).round(0).to_string())

**Third, the `loopbricks` package**, which the chapters use for anything bigger. A model is a dictionary of stocks, flows and variables, each with an equation much as in Vensim or Stella (spaces in names become underscores inside equations), and `run` returns a DataFrame indexed by time with a column for every stock, flow and variable. Settings are passed by name, so `run(dt=0.25, capacity=4100)` changes the time step and one setting for that run only.

In [ ]:
model = lb.Model.from_spec({
    "name": "Waiting list", "timeUnit": "month", "T": 24, "dt": 0.125,
    "stocks": [{"id": "wl", "name": "Waiting list", "init": "32000"}],
    "flows": [
        {"id": "ref", "name": "Referrals", "to": "wl", "eq": "Referral_rate"},
        {"id": "trt", "name": "Treatments", "from": "wl",
         "eq": "MIN(Capacity, Waiting_list / Minimum_treatment_time)"},
    ],
    "vars": [
        {"id": "rr", "name": "Referral rate", "eq": "4000"},
        {"id": "cap", "name": "Capacity", "eq": "3950"},
        {"id": "mtt", "name": "Minimum treatment time", "eq": "0.5"},
    ],
})
print(model.equations())
run = model.run()
print("\nlargest difference from plain Python:",
      (run["Waiting list"] - plain.to_numpy()).abs().max())

The three versions agree to the last digit, which is the habit the chapters build on: when you trust a simple version, use it to check the more powerful one. The package also comes with the 73 ready-made molecules the chapters use: small, reusable model structures. Below is Stock management, which chapter 8 studies: it orders goods to keep an inventory at its target while orders take time to arrive. `lb.molecule(key)` loads one, `equations()` prints it, `presets` lists its saved settings and `run(preset=...)` runs one of them.

In [ ]:
sm = lb.molecule("stockmgmt")
print(sm)
print([p["label"] for p in sm.presets])
fig, ax = plt.subplots()
for p in sm.presets[:2]:
    sm.run(preset=p["label"])["Inventory"].plot(ax=ax, label=p["label"])
ax.set(title="Stock management: two presets", ylabel="Inventory (units)")
ax.legend()
plt.show()

@sec-package covers the package and its equation language in full.

**Try it 8.**

- *Check.* Run `simulate` with `capacity=4000` and confirm the list stays at 32,000.
- *Apply.* Halve `dt` to 0.0625 in `model.run(dt=0.0625)` and measure how much the final value changes. Why so little?
- *Extend.* Use `simulate_np` with `capacity=np.linspace(3800, 4200, 41)` and plot the list at month 24 against capacity. Is the line straight or curved, and why?

<details><summary>Worked answer</summary>

```python
print(simulate(capacity=4000).iloc[-1])                  # 32000.0
a = model.run()["Waiting list"].iloc[-1]
b = model.run(dt=0.0625)["Waiting list"].iloc[-1]
print(a - b)   # 0.0: with a long list, treatments sit at capacity, so the net flow is constant
               # and Euler's method is exact for a constant flow, whatever the dt
caps = np.linspace(3800, 4200, 41)
times, many = simulate_np(capacity=caps)
fig, ax = plt.subplots(); ax.plot(caps, many[-1]); plt.show()
# A straight line: capacity binds everywhere in this range. It would bend only when the list
# got short enough (under capacity × 0.5 patients) for the list itself to limit treatments.
```

</details>

## 10. Common mistakes

| Symptom | Likely cause | Fix |
|---|---|---|
| `NameError` on a name you defined | the defining cell was not run, or was run after | restart and run all |
| results change when you rerun a cell | a cell changes a name it also reads, such as `x = x + 1` | restart and run all |
| a list or DataFrame changed when you edited a copy | `b = a` shares, it does not copy | `b = a.copy()` |
| `[1, 2] * 2` gives `[1, 2, 1, 2]` | a list, not an array | `np.array([1, 2]) * 2` |
| `x == 0.3` is `False` for a computed 0.3 | rounding in binary | `np.isclose(x, 0.3)` |
| off by one at the end of a range | `range` and slices stop *before* `stop` | `range(n + 1)`, or `linspace` |
| `KeyError` on a pandas column | a typo or a capital letter | print `df.columns` |
| `SettingWithCopyWarning` from pandas | changing a slice of a table | change it with `.loc[rows, col] = ...` on the whole table |
| a chart has everything on one line | forgot `label=` or `ax.legend()` | add both |
| a chart does not appear | the cell ended before `plt.show()` | end chart cells with `plt.show()` |

## 11. Quick reference {#sec-py-reference}

| Task | Code |
|---|---|
| name a value | `capacity = 3950` |
| print with format | `print(f"{x:,.0f} patients, {share:.0%}")` |
| list, position, slice | `xs = [1, 2, 3]`, `xs[0]`, `xs[-1]`, `xs[1:3]` |
| dictionary | `d = {"a": 1}`, `d["a"]`, `d.get("b", 0)`, `d.items()` |
| decide | `if x > 0: ... elif ...: ... else: ...` |
| loop | `for i in range(n):`, `for k, v in d.items():`, `while cond:` |
| build a list | `[f(x) for x in xs if x > 0]` |
| function | `def f(x, k=1): return k * x` |
| spread settings | `f(**{"x": 2, "k": 3})` |
| catch, check | `try: ... except KeyError as e: ...`, `assert ok, "message"` |
| array | `np.array(xs)`, `np.arange(0, 1, 0.1)`, `np.linspace(0, 1, 11)` |
| maths on arrays | `np.exp`, `np.log`, `np.minimum`, `a.mean()`, `a.sum(axis=0)` |
| select by condition | `a[a > 0]`, `np.where(a > 0, a, 0)`, `(a > 0).sum()` |
| flow from stock and back | `np.diff(s)`, `s[0] + np.cumsum(f)` |
| random, repeatable | `rng = np.random.default_rng(1)`, `rng.normal(0, 1, size=100)` |
| read data | `pd.read_csv(path, parse_dates=["month"]).set_index("month")` |
| column, rows | `df["total"]`, `df.loc["2020"]`, `df.iloc[-1]` |
| change over time | `s.diff()`, `s.shift(1)`, `s.pct_change()`, `s.rolling(12).mean()` |
| group summary | `s.groupby(s.index.year).mean()` |
| table from rows | `pd.DataFrame(list_of_dicts)` |
| chart | `fig, ax = plt.subplots()`, `ax.plot(x, y, label=...)`, `ax.set(...)`, `ax.legend()`, `plt.show()` |
| run a molecule | `lb.molecule("decay").run(dt=0.25, decay_time=3)` |

## Going further

- The official [Python tutorial](https://docs.python.org/3/tutorial/) covers the language in more depth; sections 3 to 5 match this appendix.
- Jake VanderPlas, *Python Data Science Handbook*, free online, has full chapters on NumPy, pandas and matplotlib.
- Wes McKinney, *Python for Data Analysis*, free online, is the pandas book by the author of pandas.
- The [matplotlib gallery](https://matplotlib.org/stable/gallery/) is the fastest way to find how to draw a particular chart: find a picture like the one you want and copy its code.